In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# INITIALIZATION STEPS

In [18]:
import sys
print(sys.executable)

import subprocess
subprocess.run(["pip", "show", "chromadb"], capture_output=True, text=True).stdout

c:\Users\Moirai\anaconda3\envs\Hermes\python.exe


'Name: chromadb\nVersion: 1.5.5\nSummary: Chroma.\nHome-page: https://github.com/chroma-core/chroma\nAuthor: \nAuthor-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>\nLicense: \nLocation: C:\\Users\\Moirai\\anaconda3\\envs\\Hermes\\Lib\\site-packages\nRequires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn\nRequired-by: \n'

# CHROMA DB TEST

In [19]:
import chromadb

print(chromadb.__version__)


client = chromadb.PersistentClient(path="./chroma_db")
test = client.get_or_create_collection("test")
test.add(ids=["1"], documents=["hello world"])
print(test.query(query_texts=["hello"], n_results = 1))

1.5.5
{'ids': [['1']], 'embeddings': None, 'documents': [['hello world']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None]], 'distances': [[0.4976864159107208]]}


# CHROMA STORE TEST

In [20]:
import sys
sys.path.append("../")  # lets the notebook find the rag/module

from rag.chroma import ChromaStore

store = ChromaStore()
print("Collections loaded:", list(store.collections.keys()))

c:\Users\Moirai\anaconda3\envs\Hermes\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collections loaded: ['quests', 'lore']


# INDEXER TEST

In [21]:
import sys
sys.path.append("../")

from rag.chroma import ChromaStore
from rag.indexer import Indexer

store = ChromaStore()
indexer = Indexer(store)

# add_raw test
indexer.add_raw(
    collection="quests",
    id="q_test_001",
    text="Eat Food: You are hungry. Find something to eat.",
    metadata={"source": "test"}
)
print("After add_raw:", store.count("quests"))

# add_bulk test
indexer.add_bulk("lore", [
    {"id": "l_test_001", "text": "Hokma: deez nuts", "metadata": {"source": "test"}},
    {"id": "l_test_002", "text": "Shmebulock: shmebulock!", "metadata": {"source": "test"}},
])
print("After add_bulk:", store.count("lore"))

# upsert (i.e., same id, updated text)
indexer.add_raw(
    collection="quests",
    id="q_test_001",
    text="Obtain Kromer: Consume kromer.",
    metadata={"source": "test"}
)
print("After upsert:", store.count("quests"))
print("Updated text:", store.query("quests", "Lost Sword")[0]["text"])

# cleanup 
store.delete("quests", ["q_test_001"])
store.delete("lore", ["l_test_001", "l_test_002"])
print("Quests after cleanup:", store.count("quests"))
print("Lore after cleanup:", store.count("lore"))

After add_raw: 1
After add_bulk: 2
After upsert: 1
Updated text: Obtain Kromer: Consume kromer.
Quests after cleanup: 0
Lore after cleanup: 0


# RETRIEVER TEST

In [22]:
import sys
sys.path.append("../")

from rag.chroma import ChromaStore
from rag.indexer import Indexer
from rag.retriever import Retriever

store = ChromaStore()
indexer = Indexer(store)
retriever = Retriever(store)

# test data 
indexer.add_bulk("quests", [
    {"id": "q_test_001", "text": "Eat Food: You are hungry. Find something to eat.", "metadata": {"source": "test"}},
    {"id": "q_test_002", "text": "Obtain Kromer: Consume kromer.", "metadata": {"source": "test"}},
])
indexer.add_bulk("lore", [
    {"id": "l_test_001", "text": "Hokma: deez nuts", "metadata": {"source": "test"}},
    {"id": "l_test_002", "text": "Shmebulock: shmebulock!", "metadata": {"source": "test"}},
])

# test query 
results = retriever.query("I am Hokma and I am in need of bread.")

for r in results:
    print(f"[{r['collection']}] Score: {r['score']:.2f} | {r['text']}")

# cleanup 
store.delete("quests", ["q_test_001", "q_test_002"])
store.delete("lore", ["l_test_001", "l_test_002"])
print("\nCleaned up.")

[lore] Score: 0.51 | Hokma: deez nuts
[quests] Score: 0.35 | Eat Food: You are hungry. Find something to eat.

Cleaned up.


# RAG API TEST

In [23]:
import sys
sys.path.append("../")

from rag.rag_api import RAGAPI

rag = RAGAPI()

# add single entry
rag.add_raw(
    collection="quests",
    id="q_test_001",
    text="Eat Food: You are hungry. Find something to eat.",
    metadata={"source": "test"}
)

# add bulk 
rag.add_bulk("lore", [
    {"id": "l_test_001", "text": "Hokma: deez nuts", "metadata": {"source": "test"}},
    {"id": "l_test_002", "text": "Shmebulock: shmebulock!", "metadata": {"source": "test"}},
])

# query
results = rag.query("I am hungry and need Kromer")

for r in results:
    print(f"[{r['collection']}] Score: {r['score']:.2f} | {r['text']}")

# cleanup 
rag.store.delete("quests", ["q_test_001"])
rag.store.delete("lore", ["l_test_001", "l_test_002"])
print("\nCleaned up.")

[quests] Score: 0.31 | Eat Food: You are hungry. Find something to eat.

Cleaned up.


# RETRIEVER TEST

In [24]:
import sys
sys.path.append("./")

from rag.chroma import ChromaStore
from rag.indexer import Indexer
from rag.loader import preview_folder, ingest_folder

store = ChromaStore()
indexer = Indexer(store)

# preview 
preview_folder(
    folder="./data/rawHTML",
    collection="lore"
)

# set this manually after reviewing the preview above, though this step should be available in the UI at some point
CONFIRM = False  # change to True to proceed with ingestion

if CONFIRM:
    ingest_folder(
        folder="./data/rawHTML",
        collection="lore",
        indexer=indexer,
    )
    print("Lore count:", store.count("lore"))
else:
    print("Ingestion skipped. Set CONFIRM = True to proceed.")

Found 148 file(s) to ingest into 'lore':

  ID:      Acquire_the_Gauntlets_for_Helsik
  Preview: Acquire the Gauntlets for Helsik - bg3.wiki Ad placeholder Acquire the Gauntlets for Helsik From bg3.wiki Jump to navigation Jump to search Helsik behind her counter Acquire the Gauntlets for Helsik i...

  ID:      Aid_the_Underduke
  Preview: Aid the Underduke - bg3.wiki Ad placeholder Aid the Underduke From bg3.wiki Jump to navigation Jump to search Nine-Fingers Keene wishes to protect her Guild. Aid the Underduke is a Quest in Act Three ...

  ID:      Ask_the_Goblin_Priestess_for_Help
  Preview: Ask the Goblin Priestess for Help - bg3.wiki Ad placeholder Ask the Goblin Priestess for Help From bg3.wiki Jump to navigation Jump to search Priestess Gut may be able to help with the tadpole problem...

  ID:      Avenge_Glut%27s_Circle
  Preview: Avenge Glut's Circle - bg3.wiki Ad placeholder Avenge Glut's Circle From bg3.wiki Jump to navigation Jump to search Sovereign Glut seeks revenge. A

# MANIFEST TEST

In [25]:
import sys
sys.path.append("./")

from rag.manifest import Manifest

manifest = Manifest(chroma_dir="./chroma_db")

# record a file 
manifest.record("ashenvale.html", "lore")
manifest.record("lost_sword.html", "quests")
print("After recording:", manifest.all())

# check if ingested 
print("ashenvale.html ingested?", manifest.is_ingested("ashenvale.html"))
print("unknown.html ingested?",   manifest.is_ingested("unknown.html"))

# remove one 
manifest.remove("ashenvale.html")
print("After remove:", manifest.all())

# clear everything 
manifest.clear()
print("After clear:", manifest.all())

After recording: {'ashenvale.html': {'ingested_at': '2026-03-27T18:26:35.862298', 'collection': 'lore'}, 'lost_sword.html': {'ingested_at': '2026-03-27T18:26:35.862749', 'collection': 'quests'}}
ashenvale.html ingested? True
unknown.html ingested? False
After remove: {'lost_sword.html': {'ingested_at': '2026-03-27T18:26:35.862749', 'collection': 'quests'}}
After clear: {}


# MANIFEST LOADER + API TEST

In [26]:
import sys
sys.path.append("./")

from rag.rag_api import RAGAPI

rag = RAGAPI()

# preview 
rag.preview("./data/rawHTML", "lore")

# first ingest 
CONFIRM = False  # set to True to proceed
if CONFIRM:
    rag.ingest("./data/rawHTML", "lore")
    print("\n Manifest after first ingest")
    rag.show_manifest()

    #  second ingest (should skip everything) 
    print("\n Second ingest (should skip)")
    rag.ingest("./data/rawHTML", "lore")

    # force re-ingest
    print("\n Force re-ingest")
    rag.force_reingest("./data/rawHTML", "lore")
    rag.show_manifest()

    # clear manifest 
    rag.clear_manifest()
    rag.show_manifest()

Found 148 file(s) to ingest into 'lore':

  ID:      Acquire_the_Gauntlets_for_Helsik
  Preview: Acquire the Gauntlets for Helsik - bg3.wiki Ad placeholder Acquire the Gauntlets for Helsik From bg3.wiki Jump to navigation Jump to search Helsik behind her counter Acquire the Gauntlets for Helsik i...

  ID:      Aid_the_Underduke
  Preview: Aid the Underduke - bg3.wiki Ad placeholder Aid the Underduke From bg3.wiki Jump to navigation Jump to search Nine-Fingers Keene wishes to protect her Guild. Aid the Underduke is a Quest in Act Three ...

  ID:      Ask_the_Goblin_Priestess_for_Help
  Preview: Ask the Goblin Priestess for Help - bg3.wiki Ad placeholder Ask the Goblin Priestess for Help From bg3.wiki Jump to navigation Jump to search Priestess Gut may be able to help with the tadpole problem...

  ID:      Avenge_Glut%27s_Circle
  Preview: Avenge Glut's Circle - bg3.wiki Ad placeholder Avenge Glut's Circle From bg3.wiki Jump to navigation Jump to search Sovereign Glut seeks revenge. A